# BFCN: Bio-inspired Feature Contextual Network

**Bio-Inspired Deep Learning Model**

Paper: "Contextual modulation via nCRF with multi-pathway integration"

**Bio-Inspiration**: Contextual modulation via normalized Contour Receptive Field (nCRF)  
**Deep Learning Enhancement**: Multi-pathway integration  
**Improvement Area**: Saliency and texture suppression

This notebook implements BFCN and evaluates on HED_Small dataset.

In [ ]:
# ===== Configuration =====
from pathlib import Path
import sys

PROJECT_ROOT = Path('..')
DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'HED_Small'
MODELS_DIR = PROJECT_ROOT / 'models' / 'bfcn'
OUTPUT_DIR = PROJECT_ROOT / 'bio DL' / 'outputs' / 'BFCN'

DEVICE = 'cuda'
BATCH_SIZE = 1
NUM_TEST_IMAGES = 20

In [ ]:
# ===== Imports =====
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'opencv-python', 'matplotlib', 'numpy', 'pillow', 'tqdm', 'scikit-learn'], check=False)

import os, numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device(DEVICE if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## BFCN Architecture with nCRF

nCRF (normalized Contour Receptive Field) provides contextual modulation similar to cortical surround suppression.

In [ ]:
# ===== BFCN Architecture =====
class nCRFModule(nn.Module):
    """Normalized Contour Receptive Field for contextual modulation"""
    def __init__(self, channels):
        super().__init__()
        self.center = nn.Conv2d(channels, channels, 3, padding=1)
        self.surround = nn.Conv2d(channels, channels, 5, padding=2)
        self.normalize = nn.BatchNorm2d(channels)
        
    def forward(self, x):
        center = self.center(x)
        surround = self.surround(x)
        # Surround suppression
        modulated = center - 0.5 * surround
        return F.relu(self.normalize(modulated))

class MultiPathway(nn.Module):
    """Multi-pathway integration for different scales"""
    def __init__(self, in_ch):
        super().__init__()
        self.path1 = nn.Conv2d(in_ch, in_ch, 3, padding=1)
        self.path2 = nn.Conv2d(in_ch, in_ch, 5, padding=2)
        self.path3 = nn.Conv2d(in_ch, in_ch, 7, padding=3)
        self.fusion = nn.Conv2d(in_ch * 3, in_ch, 1)
        
    def forward(self, x):
        p1 = F.relu(self.path1(x))
        p2 = F.relu(self.path2(x))
        p3 = F.relu(self.path3(x))
        return self.fusion(torch.cat([p1, p2, p3], dim=1))

class BFCN(nn.Module):
    """Bio-inspired Feature Contextual Network"""
    def __init__(self):
        super().__init__()
        
        # Encoder with nCRF
        self.enc1 = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1), nn.ReLU())
        self.ncrf1 = nCRFModule(64)
        self.mp1 = MultiPathway(64)
        
        self.enc2 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU())
        self.ncrf2 = nCRFModule(128)
        self.mp2 = MultiPathway(128)
        
        self.enc3 = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1), nn.ReLU())
        self.ncrf3 = nCRFModule(256)
        
        # Edge outputs
        self.edge1 = nn.Conv2d(64, 1, 1)
        self.edge2 = nn.Conv2d(128, 1, 1)
        self.edge3 = nn.Conv2d(256, 1, 1)
        self.fusion = nn.Conv2d(3, 1, 1)
        
    def forward(self, x):
        h, w = x.shape[2:]
        
        # Encode with nCRF and multi-pathway
        e1 = self.enc1(x)
        e1 = self.ncrf1(e1)
        e1 = self.mp1(e1)
        s1 = self.edge1(e1)
        
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e2 = self.ncrf2(e2)
        e2 = self.mp2(e2)
        s2 = self.edge2(e2)
        
        e3 = self.enc3(F.max_pool2d(e2, 2))
        e3 = self.ncrf3(e3)
        s3 = self.edge3(e3)
        
        # Upsample and fuse
        s1 = F.interpolate(s1, (h, w), mode='bilinear', align_corners=False)
        s2 = F.interpolate(s2, (h, w), mode='bilinear', align_corners=False)
        s3 = F.interpolate(s3, (h, w), mode='bilinear', align_corners=False)
        
        fused = self.fusion(torch.cat([s1, s2, s3], dim=1))
        return torch.sigmoid(fused), [torch.sigmoid(s) for s in [s1, s2, s3]]

model = BFCN().to(DEVICE)
print(f"✓ Created BFCN model ({sum(p.numel() for p in model.parameters()):,} params)")
model.eval()

In [ ]:
# ===== Dataset =====
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir = root / split / 'images'
        self.gt_dir = root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        gt_path = self.gt_dir / img_path.name.replace('.jpg', '.png')
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0 if gt_path.exists() else np.zeros((img.shape[0], img.shape[1]), dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

test_dataset = EdgeDataset(DATASET_ROOT, 'test')
if NUM_TEST_IMAGES: test_dataset.images = test_dataset.images[:NUM_TEST_IMAGES]
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Test images: {len(test_dataset)}")

In [ ]:
# ===== Metrics & Inference =====
def dilate_gt(gt, r=1):
    return cv2.dilate(gt, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1)))

def compute_metrics(preds, labels):
    threshs = np.linspace(0.05, 0.95, 30)
    all_preds, all_labels, ois_scores = [], [], []
    for pred, label in zip(preds, labels):
        label_tol = dilate_gt((label > 0.5).astype(np.float32), r=1).flatten()
        pred_smooth = cv2.GaussianBlur(pred, (3,3), 0).flatten()
        all_preds.append(pred_smooth); all_labels.append(label_tol)
        best_f1 = 0.0
        for t in threshs:
            pred_bin = (pred_smooth >= t).astype(np.float32)
            tp, fp, fn = np.sum(pred_bin * label_tol), np.sum(pred_bin * (1 - label_tol)), np.sum((1 - pred_bin) * label_tol)
            f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
            best_f1 = max(best_f1, f1)
        ois_scores.append(best_f1)
    all_preds_flat, all_labels_flat = np.concatenate(all_preds), np.concatenate(all_labels)
    best_ods, best_thresh = 0.0, 0.0
    for t in threshs:
        pred_bin = (all_preds_flat >= t).astype(np.float32)
        tp, fp, fn = np.sum(pred_bin * all_labels_flat), np.sum(pred_bin * (1 - all_labels_flat)), np.sum((1 - pred_bin) * all_labels_flat)
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
        if f1 > best_ods: best_ods, best_thresh = f1, t
    ap = average_precision_score(all_labels_flat, all_preds_flat) if np.sum(all_labels_flat) > 0 else 0.0
    return {'ODS': best_ods, 'ODS_thresh': best_thresh, 'OIS': np.mean(ois_scores), 'AP': ap}

predictions, ground_truths, image_names = [], [], []
with torch.no_grad():
    for imgs, gts, names in tqdm(test_loader, desc="Inference"):
        fuse, _ = model(imgs.to(DEVICE))
        for i in range(fuse.shape[0]):
            predictions.append(fuse[i, 0].cpu().numpy())
            ground_truths.append(gts[i].cpu().numpy())
            image_names.append(names[i])

metrics = compute_metrics(predictions, ground_truths)
print(f"\n{'='*60}\nBFCN Results\n{'='*60}")
print(f"ODS: {metrics['ODS']:.4f} | OIS: {metrics['OIS']:.4f} | AP: {metrics['AP']:.4f}\n{'='*60}")

In [ ]:
# ===== Save Results =====
import json
results = {'model': 'BFCN', 'bio_mechanism': 'nCRF contextual modulation', 'improvement':  'Saliency and texture suppression', 'metrics': metrics, 'num_test_images': len(predictions)}
with open(OUTPUT_DIR / 'bfcn_metrics.json', 'w') as f: json.dump(results, f, indent=2)
print(f"✓ Saved to {OUTPUT_DIR}\n✅ BFCN evaluation complete!")